# Slide Exercise 04: Multi-Modal Movie Recommender

This is the refined version of `MultiModal_MovieRecommender.ipynb`.

Learning objectives:
- Display real movie posters so the recommendation output is visible, not only numerical.
- Combine text features with poster/image features.
- Compare text-only and multi-modal rankings.
- Use optional CLIP image embeddings when available, with a simple fallback.

Main functions used:
- `TfidfVectorizer(...)`: creates text vectors.
- `requests.get(...)`: downloads poster images from URLs.
- `Image.open(...)`: opens downloaded poster images with Pillow.
- `plt.imshow(...)`: displays posters in the notebook.
- `SentenceTransformer('clip-ViT-B-32')`: optionally encodes poster images into image embeddings.
- `np.hstack(...)`: combines text and image/visual feature arrays.
- `cosine_similarity(...)`: ranks movies after feature fusion.


Use a small MovieLens-style subset with real poster URLs. This mirrors the earlier notebook you liked, but adds safer fallback logic and clearer teaching explanations.


In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display, HTML
from io import BytesIO
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import requests

movies = pd.DataFrame({
    "movieId": [1, 2, 3, 4, 5, 6],
    "title": [
        "Toy Story",
        "Jumanji",
        "Grumpier Old Men",
        "Waiting to Exhale",
        "Father of the Bride Part II",
        "Heat",
    ],
    "genres": [
        "Animation|Children|Comedy",
        "Adventure|Children|Fantasy",
        "Comedy|Romance",
        "Comedy|Drama|Romance",
        "Comedy",
        "Action|Crime|Thriller",
    ],
    "poster_url": [
        "https://image.tmdb.org/t/p/w500/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg",
        "https://image.tmdb.org/t/p/w500/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg",
        "https://image.tmdb.org/t/p/w500/qvktm0BHcnmDpul4Hz01GIazWPr.jpg",
        "https://image.tmdb.org/t/p/w500/sJnO6G3HXHzeT6rv2yzZbR6quT3.jpg",
        "https://image.tmdb.org/t/p/w500/3X0XKxLZ6fHf8mJ7gu9BID8K0F6.jpg",
        "https://image.tmdb.org/t/p/w500/rr7E0NoGKxvbkb89eR1GwfoYjpA.jpg",
    ],
})

movies


,movieId,title,genres,poster_url
0,1,Toy Story,Animation|Children|Comedy,https://image.tmdb.org/t/p/w500/uXDfjJbdP4ijW5...
1,2,Jumanji,Adventure|Children|Fantasy,https://image.tmdb.org/t/p/w500/vzmL6fP7aPKNKP...
2,3,Grumpier Old Men,Comedy|Romance,https://image.tmdb.org/t/p/w500/qvktm0BHcnmDpu...
3,4,Waiting to Exhale,Comedy|Drama|Romance,https://image.tmdb.org/t/p/w500/sJnO6G3HXHzeT6...
4,5,Father of the Bride Part II,Comedy,https://image.tmdb.org/t/p/w500/3X0XKxLZ6fHf8m...
5,6,Heat,Action|Crime|Thriller,https://image.tmdb.org/t/p/w500/rr7E0NoGKxvbkb...


Display the posters directly in the notebook. Even before embeddings, students can see that multi-modal recommendation means using more than text.


In [2]:
def show_poster_grid(movie_table, title="Movie posters"):
    cards = []
    for _, row in movie_table.iterrows():
        cards.append(
            f'''
            <div style="width:150px; margin:8px; display:inline-block; vertical-align:top; text-align:center;">
              <img src="{row['poster_url']}" style="width:140px; height:210px; object-fit:cover; border-radius:6px;">
              <div style="font-size:13px; margin-top:6px;">{row['title']}</div>
            </div>
            '''
        )
    display(HTML(f"<h3>{title}</h3>" + "".join(cards)))

show_poster_grid(movies)


Create text vectors from title and genres. This is the text modality.


In [3]:
movies["text"] = movies["title"] + " " + movies["genres"].str.replace("|", " ", regex=False)

tfidf = TfidfVectorizer(stop_words="english")
text_matrix = tfidf.fit_transform(movies["text"]).toarray()
text_similarity = cosine_similarity(text_matrix)

pd.DataFrame(text_matrix, index=movies["title"], columns=tfidf.get_feature_names_out()).round(2)


,action,adventure,animation,bride,children,comedy,crime,drama,exhale,fantasy,...,heat,ii,jumanji,men,old,romance,story,thriller,toy,waiting
title,,,,,,,,,,,,,,,,,,,,,
Toy Story,0.0,0.00,0.5,0.00,0.41,0.30,0.0,0.0,0.0,0.00,...,0.0,0.00,0.00,0.0,0.0,0.00,0.5,0.0,0.5,0.0
Jumanji,0.0,0.52,0.0,0.00,0.43,0.00,0.0,0.0,0.0,0.52,...,0.0,0.00,0.52,0.0,0.0,0.00,0.0,0.0,0.0,0.0
Grumpier Old Men,0.0,0.00,0.0,0.00,0.00,0.30,0.0,0.0,0.0,0.00,...,0.0,0.00,0.00,0.5,0.5,0.41,0.0,0.0,0.0,0.0
Waiting to Exhale,0.0,0.00,0.0,0.00,0.00,0.30,0.0,0.5,0.5,0.00,...,0.0,0.00,0.00,0.0,0.0,0.41,0.0,0.0,0.0,0.5
Father of the Bride Part II,0.0,0.00,0.0,0.55,0.00,0.32,0.0,0.0,0.0,0.00,...,0.0,0.55,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.0
Heat,0.5,0.00,0.0,0.00,0.00,0.00,0.5,0.0,0.0,0.00,...,0.5,0.00,0.00,0.0,0.0,0.00,0.0,0.5,0.0,0.0


Try to load real poster images. If the network is unavailable during class, the function returns a simple placeholder image instead of breaking the notebook.


In [4]:
def placeholder_image(title, size=(220, 330)):
    image = Image.new("RGB", size, color=(235, 238, 245))
    draw = ImageDraw.Draw(image)
    draw.rectangle([0, 0, size[0] - 1, size[1] - 1], outline=(120, 130, 150), width=3)
    draw.text((14, 20), title[:22], fill=(30, 40, 60))
    draw.text((14, 55), "poster unavailable", fill=(80, 90, 110))
    return image

def load_poster(url, title, timeout=8):
    try:
        response = requests.get(url, timeout=timeout)
        response.raise_for_status()
        return Image.open(BytesIO(response.content)).convert("RGB")
    except Exception as exc:
        print(f"Using placeholder for {title}: {type(exc).__name__}")
        return placeholder_image(title)

poster_images = [
    load_poster(row.poster_url, row.title)
    for row in movies.itertuples(index=False)
]

print("Loaded poster images:", len(poster_images))


Using placeholder for Toy Story: ConnectionError
Using placeholder for Jumanji: ConnectionError
Using placeholder for Grumpier Old Men: ConnectionError
Using placeholder for Waiting to Exhale: ConnectionError
Using placeholder for Father of the Bride Part II: ConnectionError
Using placeholder for Heat: ConnectionError
Loaded poster images: 6


Optional path: encode real poster images with CLIP. Fallback path: use transparent visual features extracted from the poster pixels, such as average color and brightness.


In [5]:
def normalize_rows(matrix):
    matrix = np.asarray(matrix, dtype=float)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return matrix / norms

embedding_source = "simple poster pixel features"

try:
    from sentence_transformers import SentenceTransformer
    image_model = SentenceTransformer("clip-ViT-B-32")
    image_matrix = image_model.encode(poster_images, show_progress_bar=False)
    embedding_source = "CLIP poster embeddings"
except Exception as exc:
    print("CLIP image embeddings are optional. Using simple poster pixel features instead.")
    print(type(exc).__name__, str(exc)[:160])
    rows = []
    for image in poster_images:
        small = image.resize((1, 1))
        r, g, b = np.array(small)[0, 0] / 255
        gray = image.convert("L").resize((1, 1))
        brightness = np.array(gray)[0, 0] / 255
        rows.append([r, g, b, brightness])
    image_matrix = StandardScaler().fit_transform(np.array(rows))

text_weight = 0.65
image_weight = 0.35
multimodal_matrix = np.hstack([
    text_weight * normalize_rows(text_matrix),
    image_weight * normalize_rows(image_matrix),
])

multimodal_similarity = cosine_similarity(multimodal_matrix)
embedding_source


/Users/mehrdadjalali/Library/Python/3.9/lib/python/site-packages/google/api_core/_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.9.6). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/Users/mehrdadjalali/Library/Python/3.9/lib/python/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/mehrdadjalali/Library/Python/3.9/lib/python/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google 

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


'CLIP poster embeddings'

Compare rankings. Multi-modal fusion can slightly change the order when visual style supports or weakens the text match.


In [6]:
def recommend(title, similarity_matrix, label, n=5):
    idx = movies.index[movies["title"].eq(title)][0]
    ranked = similarity_matrix[idx].argsort()[::-1]
    rows = []
    for other_idx in ranked:
        if other_idx == idx:
            continue
        rows.append({
            "method": label,
            "input_movie": title,
            "recommended_movie": movies.loc[other_idx, "title"],
            "score": round(float(similarity_matrix[idx, other_idx]), 3),
        })
        if len(rows) == n:
            break
    return pd.DataFrame(rows)

pd.concat([
    recommend("Toy Story", text_similarity, "text only"),
    recommend("Toy Story", multimodal_similarity, "text + posters"),
], ignore_index=True)


,method,input_movie,recommended_movie,score
0,text only,Toy Story,Jumanji,0.175
1,text only,Toy Story,Father of the Bride Part II,0.096
2,text only,Toy Story,Waiting to Exhale,0.087
3,text only,Toy Story,Grumpier Old Men,0.087
4,text only,Toy Story,Heat,0.000
5,text + posters,Toy Story,Jumanji,0.360
6,text + posters,Toy Story,Father of the Bride Part II,0.299
7,text + posters,Toy Story,Waiting to Exhale,0.293
8,text + posters,Toy Story,Grumpier Old Men,0.293
9,text + posters,Toy Story,Heat,0.225


Visualize the query movie and its recommendations. This is the main classroom payoff: students see the recommended items, not only a score table.


In [7]:
def show_recommendations(title, similarity_matrix, n=3):
    recs = recommend(title, similarity_matrix, "multi-modal", n=n)
    selected_titles = [title] + recs["recommended_movie"].tolist()
    selected = movies[movies["title"].isin(selected_titles)].copy()
    selected["rank"] = selected["title"].map({movie_title: i for i, movie_title in enumerate(selected_titles)})
    selected = selected.sort_values("rank")
    show_poster_grid(selected, title=f"Query and Top-{n} recommendations for {title}")
    return recs

show_recommendations("Toy Story", multimodal_similarity, n=3)


,method,input_movie,recommended_movie,score
0,multi-modal,Toy Story,Jumanji,0.360
1,multi-modal,Toy Story,Father of the Bride Part II,0.299
2,multi-modal,Toy Story,Waiting to Exhale,0.293


Interpretation:

Multi-modal systems combine evidence from more than one representation. Here students can inspect the poster images directly, then see how image features can be fused with text features. In a full environment the notebook can use CLIP embeddings; in a basic environment it still runs with simple poster pixel features.

Student task:
1. Change `image_weight` from `0.35` to `0.60`.
2. Try `show_recommendations("Heat", multimodal_similarity, n=3)`.
3. Which ranking changes are useful, and which look like visual noise?
